In [1]:
import pandas as pd
from modules.data_hourly_preprocessing import DataCleaner

In [2]:
TARGET_COL ="pm25"
HORIZON = 24

In [3]:
fecheck_df= pd.read_parquet(r"D:\pypipeline\data\processed\hourly\us_paro_hourly\test_processed.parquet")

features_exclude = [f"{TARGET_COL}_plus_{i}h" for i in range(1, HORIZON + 1)] + \
                       [f"o3_plus_{i}h" for i in range(1, HORIZON + 1)] + \
                       [f"pm25_plus_{i}h" for i in range(1, HORIZON + 1)] + \
                       ["segment_id", "imputation_confidence"]

In [4]:
fecheck_df =fecheck_df.drop(columns=features_exclude)

In [5]:
fecheck_df.head()

,pm25,o3,pm25_target,o3_target,hour,day_of_week,is_weekend,is_night,pm25_missing,o3_missing,...,o3_roll_6,pm25_roll_12,o3_roll_12,pm25_roll_24,o3_roll_24,pm25_slope_3h,o3_slope_3h,pm25_slope_12h,o3_slope_12h,pm25_o3_ratio
date,,,,,,,,,,,,,,,,,,,,,
2020-08-04 10:00:00+05:45,16.0,0.039,16.0,0.039,10,1,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,410.25641
2020-08-04 11:00:00+05:45,16.0,0.039,16.0,0.039,11,1,0,0,0,0,...,0.039,16.0,0.039,16.0,0.039,NaN,NaN,NaN,NaN,410.25641
2020-08-04 12:00:00+05:45,NaN,NaN,NaN,NaN,12,1,0,0,1,1,...,0.039,16.0,0.039,16.0,0.039,NaN,NaN,NaN,NaN,NaN
2020-08-04 13:00:00+05:45,NaN,NaN,NaN,NaN,13,1,0,0,1,1,...,0.039,16.0,0.039,16.0,0.039,NaN,NaN,NaN,NaN,NaN
2020-08-04 14:00:00+05:45,NaN,NaN,NaN,NaN,14,1,0,0,1,1,...,0.039,16.0,0.039,16.0,0.039,NaN,NaN,NaN,NaN,NaN


In [6]:
df = pd.read_csv(r"D:\pypipeline\data\raw\pred_ingestion\us_paro\us_paro_hourly.csv")

In [7]:
df.drop(columns=["pm10"], inplace=True)

In [8]:
df["date"] = pd.to_datetime(df["date"])
df.set_index("date", inplace=True)

In [9]:
clean = DataCleaner()
df_1 = clean.add_time_features(df)
df_2 = clean.add_missing_flags(df_1)
df_2["was_imputed"]=0
df_3 = clean.add_gap_length(df_2)
df_3 = clean.add_segmentation(df_3)

df_4 = clean.engineer_features(df_3)

2026-02-10 21:24:56 | INFO | modules.data_hourly_preprocessing | Adding time-based features
2026-02-10 21:24:56 | INFO | modules.data_hourly_preprocessing | Adding missing-value flags
2026-02-10 21:24:56 | INFO | modules.data_hourly_preprocessing | Computing gap length features
2026-02-10 21:24:56 | INFO | modules.data_hourly_preprocessing | Gap length features added
2026-02-10 21:24:56 | INFO | modules.data_hourly_preprocessing | Adding segment identifiers
2026-02-10 21:24:56 | INFO | modules.data_hourly_preprocessing | Segmentation complete
2026-02-10 21:24:56 | INFO | modules.data_hourly_preprocessing | Engineering features
2026-02-10 21:24:56 | INFO | modules.data_hourly_preprocessing | Adding lag features
2026-02-10 21:24:56 | INFO | modules.data_hourly_preprocessing | Adding rolling window features
2026-02-10 21:24:56 | INFO | modules.data_hourly_preprocessing | Adding slope and ratio features
2026-02-10 21:24:56 | INFO | modules.data_hourly_preprocessing | Feature engineering co

In [10]:
df_4.head()

,o3,pm25,hour,day_of_week,is_weekend,is_night,pm25_missing,o3_missing,was_imputed,pm25_gap_length,...,o3_roll_6,pm25_roll_12,o3_roll_12,pm25_roll_24,o3_roll_24,pm25_slope_3h,o3_slope_3h,pm25_slope_12h,o3_slope_12h,pm25_o3_ratio
date,,,,,,,,,,,,,,,,,,,,,
2026-02-09 17:45:00,111.87,124.68,17,0,0,0,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.114508
2026-02-09 18:45:00,99.74,130.26,18,0,0,0,0,0,0,0,...,111.8700,124.6800,111.8700,124.6800,111.8700,NaN,NaN,NaN,NaN,1.305996
2026-02-09 19:45:00,90.10,137.91,19,0,0,0,0,0,0,0,...,105.8050,127.4700,105.8050,127.4700,105.8050,6.615,-10.885,6.615,-10.885,1.530633
2026-02-09 20:45:00,83.94,145.22,20,0,0,0,0,0,0,0,...,100.5700,130.9500,100.5700,130.9500,100.5700,7.480,-7.900,6.927,-9.343,1.730045
2026-02-09 21:45:00,76.66,149.80,21,0,0,0,0,0,0,0,...,96.4125,134.5175,96.4125,134.5175,96.4125,5.945,-6.720,6.520,-8.622,1.954083
